In [1]:
import os
import uuid
import shutil
import librosa
import pandas as pd
from datetime import datetime


In [2]:
# Путь к извлечённой папке
SOURCE_DIR = "/home/merci/code/PycharmProjects/cardionix-pipeline/data/training"  # замените на свой путь
OUTPUT_DIR = "/home/merci/code/PycharmProjects/cardionix-pipeline/data/PhysioNet"

AUDIO_DIR = os.path.join(OUTPUT_DIR, "audio")
os.makedirs(AUDIO_DIR, exist_ok=True)

In [3]:
metadata = []
SOURCE_URL = "https://physionet.org/content/challenge-2016/1.0.0/"

In [4]:


for folder in os.listdir(SOURCE_DIR):
    if not folder.startswith("training-"):
        continue
    folder_path = os.path.join(SOURCE_DIR, folder)
    for fname in os.listdir(folder_path):
        if not fname.endswith(".hea"):
            continue

        base = fname.replace(".hea", "")
        wav_path = os.path.join(folder_path, base + ".wav")
        hea_path = os.path.join(folder_path, fname)

        if not os.path.exists(wav_path):
            continue

        # Считываем метку
        with open(hea_path, "r") as f:
            lines = f.readlines()
            label = next((line[1:].strip().lower() for line in lines if line.startswith("#")), "unknown")

        try:
            y, sr = librosa.load(wav_path, sr=None)
            duration = librosa.get_duration(y=y, sr=sr)
        except Exception as e:
            print(f"Ошибка при загрузке {wav_path}: {e}")
            continue

        new_uuid = str(uuid.uuid4())
        new_filename = f"{label}__{new_uuid}.wav"
        new_path = os.path.join(AUDIO_DIR, new_filename)
        shutil.copy(wav_path, new_path)

        metadata.append({
            "filename": new_filename,
            "label": label,
            "duration": duration,
            "sr": sr,
            "device": "unknown",
            "source": SOURCE_URL,
            "date": datetime.now().isoformat()
        })

# Сохраняем CSV
df = pd.DataFrame(metadata)
df

,filename,label,duration,sr,device,source,date
0,abnormal__bf372c37-8a81-40fe-83bd-ba712b6dc4f6...,abnormal,35.6660,2000,unknown,https://physionet.org/content/challenge-2016/1...,2025-06-23T18:00:08.215209
1,abnormal__c715eb88-de8e-43e7-9136-7b1e06c989ba...,abnormal,35.6660,2000,unknown,https://physionet.org/content/challenge-2016/1...,2025-06-23T18:00:08.216734
2,abnormal__419db241-d7b7-4047-9564-a2cf619bbcaf...,abnormal,21.8040,2000,unknown,https://physionet.org/content/challenge-2016/1...,2025-06-23T18:00:08.218078
3,abnormal__cc76f7b0-ce61-400c-963e-f8697f127f04...,abnormal,36.1535,2000,unknown,https://physionet.org/content/challenge-2016/1...,2025-06-23T18:00:08.219872
4,abnormal__54e87e3b-1999-482a-8507-d15cda4b93df...,abnormal,30.6505,2000,unknown,https://physionet.org/content/challenge-2016/1...,2025-06-23T18:00:08.221166
...,...,...,...,...,...,...,...
3235,normal__98302149-f913-4775-9c13-3a516f2d3b08.wav,normal,31.7760,2000,unknown,https://physionet.org/content/challenge-2016/1...,2025-06-23T18:00:14.172646
3236,normal__970e2ec1-d1f3-4ff1-a173-3309f41db0e0.wav,normal,33.2160,2000,unknown,https://physionet.org/content/challenge-2016/1...,2025-06-23T18:00:14.174175
3237,normal__30d96c95-2b08-4eba-9649-fd1a41dde25d.wav,normal,32.9280,2000,unknown,https://physionet.org/content/challenge-2016/1...,2025-06-23T18:00:14.175978
3238,abnormal__a3730b19-01a1-40ae-9894-d34185cbca0f...,abnormal,42.3360,2000,unknown,https://physionet.org/content/challenge-2016/1...,2025-06-23T18:00:14.178013


In [5]:
df.to_csv(os.path.join(OUTPUT_DIR, "annotation.csv"), index=False)